In [6]:
import pandas as pd
import numpy as np
import string
import nltk
import re
import sklearn.utils as sk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize

In [8]:
# Download NLTK data
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/salmaameer/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/salmaameer/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [10]:
# Load dataset
df = pd.read_csv('amazon_reviews.csv')
df = sk.shuffle(df)
print(df.head())

      sentiments                                     cleaned_review
13407   positive                        very nice my son loves them
9306     neutral  overall it is good keyboard but it runs out of...
13296   positive  i wanted to get replacement for my gaming setu...
10885    neutral           hurts ears and overall sound is not loud
13134   positive  good for gaming my husband likes them cancelle...


**1- Preprocessing Data**

In [11]:
def preprocess_text(text):
    ps = PorterStemmer()
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words] # remove stop words
    stemmed_tokens = [ps.stem(word) for word in filtered_tokens] # stemming

    return ' '.join(stemmed_tokens)


df['cleaned_review'] = df['cleaned_review'].fillna('') # fill missing values with empty string
df['cleaned_review'] = df['cleaned_review'].apply(preprocess_text)
print(df.head())


      sentiments                                     cleaned_review
13407   positive                                      nice son love
9306     neutral  overal good keyboard run batteri realli fast h...
13296   positive  want get replac game setup ps due wear tear ol...
10885    neutral                         hurt ear overal sound loud
13134   positive  good game husband like cancel nois good plu li...


**2- Labeling**

In [12]:
def mappingToNumbers():
  categories = {"negative": 0, "neutral": 1, "positive": 2}
  df['sentiments'] = df['sentiments'].map(categories)
  print(df.head())

mappingToNumbers()

       sentiments                                     cleaned_review
13407           2                                      nice son love
9306            1  overal good keyboard run batteri realli fast h...
13296           2  want get replac game setup ps due wear tear ol...
10885           1                         hurt ear overal sound loud
13134           2  good game husband like cancel nois good plu li...


**3- Data splitting**

In [13]:
X_train, X_test, y_train, y_test = train_test_split(df['cleaned_review'], df['sentiments'], test_size=0.2, random_state=42)

**4- TF-IDF vectorizer**

In [14]:
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

**5.1: SVM Model**

In [15]:
svmModel = SVC()
svmModel.fit(X_train, y_train)
yPred = svmModel.predict(X_test)
print("SVM Classification Report:\n", classification_report(y_test, yPred))

SVM Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.55      0.69       301
           1       0.84      0.87      0.85      1290
           2       0.91      0.94      0.93      1877

    accuracy                           0.88      3468
   macro avg       0.89      0.79      0.82      3468
weighted avg       0.88      0.88      0.88      3468



**5.2: logistic regression**

In [16]:
logisticRegModel = LogisticRegression()
logisticRegModel.fit(X_train, y_train)
yPred = logisticRegModel.predict(X_test)
print("Logistic Regression Classification Report:\n", classification_report(y_test, yPred))

Logistic Regression Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.34      0.47       301
           1       0.77      0.84      0.80      1290
           2       0.88      0.92      0.90      1877

    accuracy                           0.84      3468
   macro avg       0.82      0.70      0.72      3468
weighted avg       0.83      0.84      0.83      3468



**5.3: Naïve Bayes**

In [17]:
naiveBayesModel = MultinomialNB()

naiveBayesModel.fit(X_train, y_train)
yPred = naiveBayesModel.predict(X_test)
print("Naive Bayes Classification Report:\n", classification_report(y_test, yPred))

Naive Bayes Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.02      0.03       301
           1       0.65      0.50      0.56      1290
           2       0.71      0.93      0.80      1877

    accuracy                           0.69      3468
   macro avg       0.78      0.48      0.47      3468
weighted avg       0.71      0.69      0.65      3468



In [18]:

def newReviewLabel(vectorizer, model, newReview):
    newReview = preprocess_text(newReview) #preprocess the new review

    vect_review = vectorizer.transform([newReview]) # vectorize it using the fitted vectorizer

    reviewLabel = model.predict(vect_review)[0]

    categories = { 0: "negative", 1: "neutral", 2: "positive"}
    strLabel = categories[reviewLabel]

    return strLabel


# we used svm model because it is the highest accuracy
print(newReviewLabel(vectorizer,svmModel,"excellent for price"))
print(newReviewLabel(vectorizer,svmModel,"easy to use"))
print(newReviewLabel(vectorizer,svmModel,"the center scroll wheel broke in weeks for no reason mouse was not dropped and barely used garbage so you get what you pay for"))
print(newReviewLabel(vectorizer,svmModel,"i love it and my friends use it from me"))
print(newReviewLabel(vectorizer,svmModel,"it has a defects while using it but useful  "))
print(newReviewLabel(vectorizer,svmModel,"the product broked my back instead of recover it's very bad"))




positive
neutral
negative
positive
neutral
negative
